In [0]:
# Databricks notebook source

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    when,
    lit,
    current_timestamp
)

In [0]:
# COMMAND ----------

BRONZE_TABLE = (
    "personal.finance.bronze_transactions"
)

SILVER_TABLE = (
    "personal.finance.silver_transactions"
)

In [0]:
# COMMAND ----------

df_bronze = spark.table(
    BRONZE_TABLE
)

print(
    f"Bronze rows: {df_bronze.count()}"
)

In [0]:
# COMMAND ----------

df_silver = (
    df_bronze

    # Standardize transaction type
    .withColumn(
        "transaction_type",
        upper(
            trim(
                col("transaction_type")
            )
        )
    )

    # Standardize category
    .withColumn(
        "category",
        trim(col("category"))
    )

    # Standardize description
    .withColumn(
        "description",
        trim(col("description"))
    )

    # Standardize remark
    .withColumn(
        "remark",
        trim(col("remark"))
    )
)

In [0]:
# COMMAND ----------

df_silver = (
    df_silver

    # Transaction date must exist
    .filter(
        col("transaction_date").isNotNull()
    )

    # Amount must exist
    .filter(
        col("amount").isNotNull()
    )

    # Amount should not be negative
    .filter(
        col("amount") >= 0
    )

    # Only valid transaction types
    .filter(
        col("transaction_type").isin(
            "INCOME",
            "EXPENSE"
        )
    )
)

In [0]:
# COMMAND ----------

dedup_columns = [
    "transaction_date",
    "amount",
    "description",
    "category",
    "remark",
    "transaction_type"
]

df_silver = (
    df_silver
    .dropDuplicates(dedup_columns)
)

In [0]:
# COMMAND ----------

df_silver = (
    df_silver
    .withColumn(
        "silver_processed_timestamp",
        current_timestamp()
    )
)

In [0]:
# COMMAND ----------

silver_columns = [
    "transaction_date",
    "amount",
    "description",
    "category",
    "remark",
    "year",
    "month",
    "month_date",
    "transaction_type",
    "source",
    "ingestion_timestamp",
    "silver_processed_timestamp"
]

df_silver = df_silver.select(
    *silver_columns
)

In [0]:
# COMMAND ----------

(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(SILVER_TABLE)
)

print(
    "Silver table created successfully."
)

In [0]:
# COMMAND ----------

print(
    f"Silver rows: "
    f"{spark.table(SILVER_TABLE).count()}"
)

display(
    spark.table(SILVER_TABLE)
    .orderBy(
        col("transaction_date").desc()
    )
    .limit(20)
)